## Hardware Check

In [ ]:
!nvidia-smi
import torch
print(f'\nPyTorch version: {torch.__version__}')
print(f'CUDA available: {torch.cuda.is_available()}')
if torch.cuda.is_available():
    print(f'CUDA device: {torch.cuda.get_device_name(0)}')
    print(f'GPU memory: {torch.cuda.get_device_properties(0).total_memory / 1e9:.2f} GB')

## Setup Environment
Run this cell to clone the repository securely and install dependencies.

In [ ]:
import os
import getpass
from pathlib import Path

GITHUB_USER = 'sattary'
REPO_NAME = 'ali_proj'
BRANCH = 'fix-review'  # Update if on a different branch
PROJECT_DIR = 'ali_proj'

if not Path(PROJECT_DIR).exists():
    print('Enter your GitHub Personal Access Token (PAT):')
    PAT = getpass.getpass()
    REPO_URL = f'https://{PAT}@github.com/{GITHUB_USER}/{REPO_NAME}.git'
    print(f'Cloning {REPO_NAME} (branch: {BRANCH})...')
    os.system(f'git clone -b {BRANCH} {REPO_URL}')
else:
    print('Repository already cloned. Pulling latest changes...')
    os.system(f'cd {PROJECT_DIR} && git pull origin {BRANCH}')

os.chdir(PROJECT_DIR)

print('\nInstalling uv...')
os.system('pip install -q uv')

os.environ['MPLBACKEND'] = 'Agg'
print('\nSyncing dependencies...')
os.system('uv sync')
print('\n\u2713 Setup complete!')

## Data Directory and Configuration
Ensure your dataset and `best_config.yaml` (from Optuna) are uploaded if starting a new cloud instance.

In [ ]:
DATA_DIR = 'data/training_set'
CONFIG_FILE = 'runs/optuna/best_config.yaml'

import os
config_arg = f'--config {CONFIG_FILE}' if os.path.exists(CONFIG_FILE) else ''
if not config_arg:
    print('Warning: best_config.yaml not found. Using default architecture.')

## Phase 3: Train Primary Network

In [ ]:
EPOCHS = 100
BATCH_SIZE = 32

!uv run phase-unwrap train \
    --use-amp \
    {config_arg} \
    --run-name exp_primary \
    --data-dir {DATA_DIR} \
    --batch-size {BATCH_SIZE} \
    --epochs {EPOCHS}

print('\n\u2713 Primary training complete!')

## Phase 4: Statistical Validation (Multiseed)

In [ ]:
!uv run phase-unwrap multiseed \
    --use-amp \
    {config_arg} \
    --run-name exp_multiseed \
    --num-seeds 3 \
    --data-dir {DATA_DIR} \
    --epochs {EPOCHS} \
    --batch-size {BATCH_SIZE}

print('\n\u2713 Multiseed validation complete!')

## Phase 5: Architectural Ablation

In [ ]:
!uv run phase-unwrap ablation \
    --use-amp \
    {config_arg} \
    --out-table results/tables/ablation.tex \
    --data-dir {DATA_DIR} \
    --batch-size {BATCH_SIZE} \
    --epochs {EPOCHS}

print('\n\u2713 Ablation study complete!')

## Download Results

In [ ]:
import shutil
print('Zipping runs and results...')
shutil.make_archive('training_results', 'zip', 'runs/')
shutil.make_archive('tables_results', 'zip', 'results/')
print('\u2713 Done! You can now download training_results.zip and tables_results.zip')